# Завдання 2 · Частина 3 — Перевірка гіпотез про продуктові метрики

Проведіть серію статистичних тестів для продуктових метрик із файлу `app_metrics.csv`. Для кожного тесту наведіть дві гіпотези, формулу статистики з числовою підстановкою, критичну область, p-значення, статистичне рішення та бізнес-висновок. Використовуйте рівень значущості $\alpha=0.05$. У пункті 3(д) замість ручного обчислення статистики вкажіть використану вбудовану функцію `statsmodels`.

Джерело: [assignment_2.pdf](../input/assignment_2.pdf) (в оригіналі — «Задача 4. Інференційна статистика»).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

TASK_DIR = Path.cwd()
if (TASK_DIR / "labs" / "task2").is_dir():
    TASK_DIR = TASK_DIR / "labs" / "task2"
elif TASK_DIR.name == "notebooks":
    TASK_DIR = TASK_DIR.parent
INPUT_DIR = TASK_DIR / "input"
PDF_DIR = TASK_DIR / "pdf"
FIGURES_DIR = PDF_DIR / "figures"
SEED = 42
ALPHA = 0.05
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PDF_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})

pd.set_option("display.float_format", "{:.6g}".format)


In [2]:
# Source: the app_metrics.csv supplied alongside assignment_2.pdf.
# Keep the source file unchanged; no external download is required.
DATA_PATH = INPUT_DIR / "app_metrics.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Place the supplied app_metrics.csv at {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
required = {"Platform", "Session_Time", "Registration_Success", "Page_Views",
            "Purchase_Amount", "Revenue", "Subscription"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
for column in ("Registration_Success", "Subscription"):
    if not df[column].dropna().isin([0, 1]).all():
        raise ValueError(f"{column} must contain binary values 0 or 1")
# Review missing values before choosing a per-analysis handling policy.
display(df.head())
display(df.isna().sum().rename("missing"))

from statsmodels.stats.diagnostic import lilliefors
from statsmodels.stats.proportion import proportions_ztest


,User_ID,Platform,Session_Time,Registration_Success,Page_Views,Purchase_Amount,Revenue,Subscription
0,1001,Android,14.04,0,12,510.9,376.11,0
1,1002,Android,9.24,0,7,390.84,467.01,0
2,1003,iOS,17.76,0,13,640.27,228.87,0
3,1004,iOS,16.13,0,4,238.35,420.25,0
4,1005,Android,23.45,1,12,582.25,576.61,0


User_ID                 0
Platform                0
Session_Time            0
Registration_Success    0
Page_Views              0
Purchase_Amount         0
Revenue                 0
Subscription            0
Name: missing, dtype: int64

## 3(а). Час утримання користувача

Продакт-менеджер стверджує, що очікуваний час утримання користувача `Session_Time` перевищує 15 хвилин. Перевірте це твердження відповідним одностороннім параметричним критерієм при $\alpha=0.05$. Наведіть гіпотези, формулу статистики з числовою підстановкою, критичну область, p-значення, рішення та бізнес-висновок.

In [3]:
session = df["Session_Time"].dropna()
n_session = len(session)
session_mean, session_sd = session.mean(), session.std(ddof=1)
t_session = (session_mean - 15) / (session_sd / np.sqrt(n_session))
session_df = n_session - 1
session_critical = stats.t.ppf(1 - ALPHA, session_df)
session_test = stats.ttest_1samp(session, popmean=15, alternative="greater")
assert np.isclose(t_session, session_test.statistic)
session_results = pd.DataFrame({"n": [n_session], "mean": [session_mean], "sd": [session_sd],
    "t": [t_session], "df": [session_df], "critical": [session_critical],
    "p": [session_test.pvalue], "reject": [session_test.pvalue < ALPHA]})
display(session_results)
session_results.to_csv(RESULTS_DIR / "part3_session.csv", index=False)

,n,mean,sd,t,df,critical,p,reject
0,250,16.5201,5.00082,4.80626,249,1.651,1.33058e-06,True


***Answer***

**Гіпотези:** $H_0:\mu\le15$ проти $H_1:\mu>15$; статистику обчислюємо на межі $\mu_0=15$. Дані містять $n=250$ значень без пропусків. Припускаємо незалежну репрезентативну вибірку зі скінченною дисперсією. Точний t-критерій вимагає нормальності; при великому $n$ використовуємо наближення для середнього.

**Статистика та критична область:**
$$t=\frac{\bar x-15}{s/\sqrt n}=\frac{16.52012-15}{5.000818/\sqrt{250}}=4.806255,\qquad df=249.$$
Відхиляємо $H_0$, якщо $t>t_{0.95,249}=1.650996$.

**Результат:** $p=1.3306\times 10^{-6}<0.05$, тому $H_0$ відхиляємо. Є статистичні підстави вважати, що середній час сесії перевищує 15 хвилин. У вибірці середній час дорівнює 16.52012 хв, тобто приблизно на 1.52 хв більше за поріг. Висновок стосується середнього, а не кожного користувача, і спирається на припущення щодо вибірки.

## 3(б). Порівняння середньої суми чеку між платформами

Перевірте, чи відрізняється середній `Revenue` користувачів iOS від середнього `Revenue` користувачів Android. Генеральні дисперсії невідомі та не обов'язково рівні: використайте двосторонній t-критерій Уелша при $\alpha=0.05$. Наведіть гіпотези, формулу статистики з числовою підстановкою, критичну область, p-значення, рішення та бізнес-висновок.

In [4]:
ios = df.loc[df["Platform"] == "iOS", "Revenue"].dropna()
android = df.loc[df["Platform"] == "Android", "Revenue"].dropna()
n_ios, n_android = len(ios), len(android)
mean_ios, mean_android = ios.mean(), android.mean()
var_ios, var_android = ios.var(ddof=1), android.var(ddof=1)
v_ios, v_android = var_ios / n_ios, var_android / n_android
welch_se = np.sqrt(v_ios + v_android)
t_welch = (mean_ios - mean_android) / welch_se
welch_df = (v_ios + v_android)**2 / (v_ios**2 / (n_ios - 1) + v_android**2 / (n_android - 1))
welch_critical = stats.t.ppf(1 - ALPHA / 2, welch_df)
welch_test = stats.ttest_ind(ios, android, equal_var=False, alternative="two-sided")
assert np.isclose(t_welch, welch_test.statistic)
welch_results = pd.DataFrame({"n_ios": [n_ios], "n_android": [n_android],
    "mean_ios": [mean_ios], "mean_android": [mean_android],
    "var_ios": [var_ios], "var_android": [var_android],
    "t": [t_welch], "df": [welch_df], "critical": [welch_critical],
    "p": [welch_test.pvalue], "reject": [welch_test.pvalue < ALPHA]})
display(welch_results.T)
welch_results.to_csv(RESULTS_DIR / "part3_welch.csv", index=False)

,0
n_ios,110
n_android,140
mean_ios,500.922
mean_android,393.508
var_ios,65010.5
var_android,17623.8
t,4.01175
df,154.869
critical,1.9754
p,9.35453e-05


***Answer***

**Гіпотези:** $H_0:\mu_I=\mu_A$, $H_1:\mu_I\ne\mu_A$, де $I$ позначає iOS, $A$ - Android. Використано 110 та 140 спостережень відповідно, пропусків немає. Припускаємо незалежність користувачів і груп та скінченні дисперсії; рівності дисперсій не вимагаємо. Для ненормальних сукупностей висновок є наближеним; обидві групи досить великі для застосування ЦГТ за цих умов.

**Статистика Уелша:**
$$t=\frac{\bar x_I-\bar x_A}{\sqrt{s_I^2/n_I+s_A^2/n_A}}
=\frac{500.9221-393.5084}{\sqrt{65010.5061/110+17623.7712/140}}=4.011751.$$
Число ступенів свободи оцінено за формулою
$$\nu=\frac{(s_I^2/n_I+s_A^2/n_A)^2}{(s_I^2/n_I)^2/(n_I-1)+(s_A^2/n_A)^2/(n_A-1)}=154.869447.$$
Відхиляємо $H_0$, якщо $|t|>t_{0.975,\nu}=1.975400$.

**Результат:** $p=9.3545\times 10^{-5}<0.05$, тому $H_0$ відхиляємо. Дані свідчать про відмінність середнього Revenue між платформами. Середнє iOS вище приблизно на 107.41 грошових одиниць. Валюту в умові не вказано. Цей зв’язок не доводить, що саме платформа спричиняє різницю: групи можуть відрізнятися складом користувачів. Перевірка нормальності сукупного Revenue у пункті 3(д) не є окремою перевіркою нормальності кожної групи.

## 3(в). Коефіцієнт лінійної кореляції Пірсона

Перевірте гіпотезу про відсутність зв'язку між кількістю переглядів `Page_Views` і сумою покупки `Purchase_Amount` за допомогою критерію для коефіцієнта лінійної кореляції Пірсона при $\alpha=0.05$. Наведіть гіпотези, формулу статистики з числовою підстановкою, критичну область, p-значення, рішення та бізнес-висновок.

In [5]:
paired = df[["Page_Views", "Purchase_Amount"]].dropna()
n_pairs = len(paired)
x, y = paired["Page_Views"], paired["Purchase_Amount"]
dx, dy = x - x.mean(), y - y.mean()
sum_xy, sum_xx, sum_yy = (dx * dy).sum(), (dx**2).sum(), (dy**2).sum()
r = sum_xy / np.sqrt(sum_xx * sum_yy)
t_correlation = r * np.sqrt((n_pairs - 2) / (1 - r**2))
correlation_critical = stats.t.ppf(1 - ALPHA / 2, n_pairs - 2)
correlation_test = stats.pearsonr(x, y)
assert np.isclose(r, correlation_test.statistic)
assert np.isclose(2 * stats.t.sf(abs(t_correlation), n_pairs - 2), correlation_test.pvalue)
correlation_results = pd.DataFrame({"n": [n_pairs], "sum_xy": [sum_xy],
    "sum_xx": [sum_xx], "sum_yy": [sum_yy], "r": [r], "t": [t_correlation],
    "df": [n_pairs - 2], "critical": [correlation_critical],
    "p": [correlation_test.pvalue], "reject": [correlation_test.pvalue < ALPHA]})
display(correlation_results.T)
correlation_results.to_csv(RESULTS_DIR / "part3_correlation.csv", index=False)

,0
n,250
sum_xy,88681.7
sum_xx,1940.96
sum_yy,4.9485e+06
r,0.904875
t,33.4761
df,248
critical,1.96958
p,5.74221e-94
reject,True


***Answer***

**Гіпотези:** $H_0:\rho=0$, $H_1:\rho\ne0$. Перевіряємо саме лінійний зв’язок. Використано 250 пар Page_Views та Purchase_Amount; пропусків немає. Якби вони були, вилучалися б цілі пари. Припускаємо незалежність пар. Звичайна формула p-значення спирається на спільний нормальний розподіл двох ознак. Page_Views є кількістю переглядів, тому ця умова не виконується точно; p-значення використовуємо як наближення.

**Статистика:**
$$r=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sqrt{\sum_i(x_i-\bar x)^2\sum_i(y_i-\bar y)^2}}
=\frac{88681.7496}{\sqrt{1940.9640\cdot4948498.8700}}=0.904875,$$
$$t=r\sqrt{\frac{n-2}{1-r^2}}=0.904875\sqrt{\frac{248}{1-0.904875^2}}=33.476102.$$
За $df=248$ критична область: $|t|>t_{0.975,248}=1.969576$.

**Результат:** $p=5.7422\times 10^{-94}<0.05$, тому за використаним критерієм $H_0$ відхиляємо. Спостерігаємо сильний додатний лінійний зв’язок: користувачі з більшою кількістю переглядів зазвичай мають вищу суму покупки. Це не доводить причинного впливу переглядів на покупку. Нульова кореляція також не означала б загальної незалежності.

## 3(г). Частка користувачів із платною підпискою

Перевірте, чи генеральна частка користувачів із платною підпискою `Subscription` статистично значущо перевищує 10%. Використайте $\alpha=0.05$. Наведіть гіпотези, формулу статистики з числовою підстановкою, критичну область, p-значення, рішення та бізнес-висновок.

In [6]:
subscription = df["Subscription"].dropna()
n_subscription = len(subscription)
count_subscription = int(subscription.sum())
p0 = 0.10
subscription_share = count_subscription / n_subscription
z_subscription = (subscription_share - p0) / np.sqrt(p0 * (1 - p0) / n_subscription)
z_builtin, p_subscription = proportions_ztest(count_subscription, n_subscription,
    value=p0, alternative="larger", prop_var=p0)
assert np.isclose(z_subscription, z_builtin)
subscription_critical = stats.norm.ppf(1 - ALPHA)
subscription_results = pd.DataFrame({"n": [n_subscription], "successes": [count_subscription],
    "share": [subscription_share], "expected_successes_null": [n_subscription * p0],
    "expected_failures_null": [n_subscription * (1 - p0)], "z": [z_subscription],
    "critical": [subscription_critical], "p": [p_subscription], "reject": [p_subscription < ALPHA]})
display(subscription_results.T)
subscription_results.to_csv(RESULTS_DIR / "part3_subscription.csv", index=False)

,0
n,250
successes,34
share,0.136
expected_successes_null,25
expected_failures_null,225
z,1.89737
critical,1.64485
p,0.0288898
reject,True


***Answer***

**Гіпотези:** $H_0:p\le0.10$ проти $H_1:p>0.10$. Маємо 250 бінарних спостережень без пропусків та 34 підписки, тому $\widehat p=34/250=0.136$. Припускаємо незалежні випробування Бернуллі зі спільною ймовірністю успіху. Для нормального наближення на межі $H_0$ перевіряємо $np_0=25$ та $n(1-p_0)=225$; обидва значення перевищують 10.

**Статистика:**
$$z=\frac{\widehat p-p_0}{\sqrt{p_0(1-p_0)/n}}
=\frac{0.136-0.10}{\sqrt{0.10\cdot0.90/250}}=1.897367.$$
У `proportions_ztest` явно задано `prop_var=0.10`, щоб використати дисперсію за $H_0$. Критична область: $z>z_{0.95}=1.644854$.

**Результат:** одностороннє $p=0.028890<0.05$, тому $H_0$ відхиляємо. Дані підтримують твердження, що генеральна частка платних підписок перевищує 10%. Вибіркова частка дорівнює 13.6%, тобто на 3.6 відсоткового пункту вище за поріг. Це наближений висновок за припущень тесту; нормально розподілені окремі бінарні спостереження не потрібні.

## 3(д). Перевірка нормальності Revenue

Перевірте нормальність метрики `Revenue` одновибірковим критерієм Колмогорова--Смирнова в модифікації Лілієфорса. Використайте виключно вбудовану функцію `statsmodels`, без ручного обчислення статистики, при $\alpha=0.05$. Наведіть гіпотези, отриману статистику, p-значення та висновок щодо припущень попередніх тестів.

In [7]:
revenue = df["Revenue"].dropna()
revenue_d, revenue_p = lilliefors(revenue, dist="norm", pvalmethod="table")
revenue_results = pd.DataFrame({"n": [len(revenue)], "D": [revenue_d],
    "p_returned": [revenue_p], "reject": [revenue_p < ALPHA]})
display(revenue_results)
revenue_results.to_csv(RESULTS_DIR / "part3_revenue_normality.csv", index=False)

,n,D,p_returned,reject
0,250,0.146746,0.001,True


***Answer***

**Гіпотези:** $H_0$: Revenue має нормальний розподіл з невідомими параметрами; $H_1$: розподіл відрізняється від нормального. Використано 250 спостережень без пропусків; припускаємо незалежність і однаковий розподіл спостережень.

**Вбудована функція:** `statsmodels.stats.diagnostic.lilliefors(revenue, dist="norm", pvalmethod="table")`. Статистику вручну не обчислюємо, як вимагає цей підпункт. Отримано $D=0.146746$ та $p_{\mathrm{returned}}=0.001$. Це нижня таблична межа, тому записуємо $p\le0.001$. Правило відхилення: $p<0.05$.

**Результат:** нормальність сукупного Revenue відхиляємо. Просту нормальну модель розподілу індивідуального доходу ці дані не підтримують. Проте це не автоматично скасовує всі попередні висновки. Критерії для середніх можуть спиратися на ЦГТ при великих незалежних вибірках зі скінченною дисперсією; для Уелша важливі розподіли всередині платформ. Критерій Пірсона має окремі вимоги до спільного розподілу двох ознак. Критерій для частки використовує бінарні дані та нормальне наближення частки, а не нормальність окремих значень. Твердження умови про обов’язкову нормальність усіх ознак потребує цього уточнення.

Усі перевірки частини 3 виконано окремо при $\alpha=0.05$, як задано. Спільний рівень помилки для всієї серії тестів не контролюється.